# Initialization
Turning the symlinks into proper paths the system can access. Finding the 8mixmux8 firmware.

In [1]:
import os
from pathlib import Path

base = Path("/home/xilinx/jupyter_notebooks/qick/firmware/projects/qick_tprocv2_111_8mixmux16")
out_dir = base / "out"

# 1. Clean up the non-working links
for name in ["qick_111.bit", "qick_111.hwh"]:
    f = out_dir / name
    if f.exists() or f.is_symlink():
        f.unlink()

# 2. Create actual, native OS-level relative symlinks
os.symlink("../top/top.runs/impl_1/d_1_wrapper.bit", str(out_dir / "qick_111.bit"))
os.symlink("../top/top.gen/sources_1/bd/d_1/hw_handoff/d_1.hwh", str(out_dir / "qick_111.hwh"))

print("Native Linux symlinks successfully recreated!")

Native Linux symlinks successfully recreated!


In [2]:
from qick import QickSoc

# Load the firmware bitstream
soc = QickSoc("/home/xilinx/jupyter_notebooks/qick/firmware/projects/qick_tprocv2_111_8mixmux16/out/qick_111.bit", force_init_clks=True)

soccfg = soc

# Print to check soc details and show code has run
print(soc)

configuring reference clock chips, as requested
LMK04208 clock reference = 122.880 MHz, LMX2594 clock synth = 204.800 MHz
QICK running on ZCU111, software version 0.2.418

Firmware configuration (built Tue Jun 30 17:47:55 2026):

	Global clocks (MHz): tProc dispatcher timing 384.000, RF reference 204.800
	Groups of related clocks: [tProc core clock, tProc timing clock, DAC tile 0, DAC tile 1], [ADC tile 0, ADC tile 2]

	8 signal generator channels:
	0:	axis_sg_mixmux16_v1 - fs=6144.000 Msps, fabric=384.000 MHz
		32-bit DDS, range=768.000 MHz
		DAC tile 0, blk 0 is DAC228_T0_CH0, or RF board DAC port 0
	1:	axis_sg_mixmux16_v1 - fs=6144.000 Msps, fabric=384.000 MHz
		32-bit DDS, range=768.000 MHz
		DAC tile 0, blk 1 is DAC228_T0_CH1, or RF board DAC port 1
	2:	axis_sg_mixmux16_v1 - fs=6144.000 Msps, fabric=384.000 MHz
		32-bit DDS, range=768.000 MHz
		DAC tile 0, blk 2 is DAC228_T0_CH2, or RF board DAC port 2
	3:	axis_sg_mixmux16_v1 - fs=6144.000 Msps, fabric=384.000 MHz
		32-bit DDS, ra

In [3]:
# Jupyter setup boilerplate
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

from qick import *
from qick.asm_v2 import AveragerProgramV2 # still necessary even after importing all from QICK

# useful libraries
import numpy as np
import random 
import time
import math

# MixMuxed Signal Generation - Long Period For Spectrum Analysis

Similiar to code from Sho's Github, and copied from MixMux8SG.ipynb but modified. Implementing / testing the multi-tone generators. Readout is declared for the sake of running but is not used.

In [5]:
# --- Main MixMux16 program ---
class MixMuxProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        ro_ch = cfg['ro_ch']
        gen_ch = cfg['gen_ch']
        
        self.declare_gen(ch=gen_ch, nqz=1, ro_ch=ro_ch, 
                         mixer_freq=cfg['mixer_freq'],
                         mux_freqs=cfg['pulse_freqs'], 
                         mux_gains=cfg['pulse_gains'], 
                         mux_phases=cfg['pulse_phases'])
        
        # Declaring unused dynamic readout
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])
        self.add_readoutconfig(ch=ro_ch, name="myro", freq=cfg['pulse_freqs'][0], phase=cfg['ro_phase'], gen_ch=gen_ch)

        self.add_pulse(ch=gen_ch, name="mymixmux", 
                       style="const", 
                       length=cfg["pulse_len"],
                       mask=cfg['pulse_mask'], # tones to play must be in the mask
                      )
        
    def _body(self, cfg):        
        self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'], ddr4=True) # trigger readouts (unused)

        self.pulse(ch=cfg['gen_ch'], name="mymixmux", t=0) # trigger generator

In [9]:
gen_ch = 0 # which gen_ch to use

# --- Config dictionary ---
config = {'gen_ch': gen_ch,
          'ro_ch': 0, # unused readout channel
          'mixer_freq': 0,
          
          'pulse_freqs': [199.2, 199.3, 199.4, 199.5, 199.6, 199.7, 199.8, 199.9, 
                          200.0, 200.1, 200.2, 200.3, 200.4, 200.5, 200.6, 200.7], 
          'pulse_gains': [0.0625]*16,
          'pulse_phases': [0]*16,
          'pulse_mask': [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15],
          
          'ro_phase': 0.0,
          'trig_time': 0.4,
          'pulse_len': 4000000.0,
          'ro_len': 1.9,
         }

# --- Execution ---
prog = MixMuxProgram(soccfg, reps=1, final_delay=0.5, cfg=config)
iq_list = prog.acquire_decimated(soc, rounds=1)

# # --- Confirm completion and list frequencies ---
# print('Pulses should have successfully ran. Desired tones:')
# for i in config['pulse_freqs']:
#     print(i)

  0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
soc.reset_gens()

In [14]:
print(soccfg)

QICK running on ZCU111, software version 0.2.418

Firmware configuration (built Tue Jun 30 17:47:55 2026):

	Global clocks (MHz): tProc dispatcher timing 384.000, RF reference 204.800
	Groups of related clocks: [tProc core clock, tProc timing clock, DAC tile 0, DAC tile 1], [ADC tile 0, ADC tile 2]

	8 signal generator channels:
	0:	axis_sg_mixmux16_v1 - fs=6144.000 Msps, fabric=384.000 MHz
		32-bit DDS, range=768.000 MHz
		DAC tile 0, blk 0 is DAC228_T0_CH0, or RF board DAC port 0
	1:	axis_sg_mixmux16_v1 - fs=6144.000 Msps, fabric=384.000 MHz
		32-bit DDS, range=768.000 MHz
		DAC tile 0, blk 1 is DAC228_T0_CH1, or RF board DAC port 1
	2:	axis_sg_mixmux16_v1 - fs=6144.000 Msps, fabric=384.000 MHz
		32-bit DDS, range=768.000 MHz
		DAC tile 0, blk 2 is DAC228_T0_CH2, or RF board DAC port 2
	3:	axis_sg_mixmux16_v1 - fs=6144.000 Msps, fabric=384.000 MHz
		32-bit DDS, range=768.000 MHz
		DAC tile 0, blk 3 is DAC228_T0_CH3, or RF board DAC port 3
	4:	axis_sg_mixmux16_v1 - fs=6144.000 Msps, f

In [16]:
print(prog)                      # or prog.dump_prog() depending on version
# and check the generator got a sane envelope/freqs:
for f in config['pulse_freqs']:
    print(f, '->', soccfg.adcfreq(f, gen_ch=0))  # actual programmed grid freq

macros:
	WriteReg(dst='s_core_w1', src=0)
	Delay(t=1.0, auto=True, gens=True, ros=True, tag=None, t_params={'t': <qick.asm_v2.QickParam object at 0xffff7a50d660>}, t_regs={'t': 384})
	OpenLoop(n=1, name='reps')
	Trigger(ros=[0], tts=[], pins=[0], t=0.4, width=0.026041666666666668, ddr4=True, mr=False, tag=None, t_params={'t': <qick.asm_v2.QickParam object at 0xffff7a50ffd0>, 'width': <qick.asm_v2.QickParam object at 0xffff7a50c460>}, t_regs={'t': 154, 'width': 10}, outdict=defaultdict(<class 'int'>, {}), trigset={0, 9, 10})
	Pulse(ch=0, name='mymixmux', t=0, tag=None, t_params={'t': <qick.asm_v2.QickParam object at 0xffff7a50c1c0>}, t_regs={'t': 0})
	Wait(t=0, auto=True, gens=False, ros=True, tag=None, no_warn=True, t_params={'t': <qick.asm_v2.QickParam object at 0xffff7a50d540>}, t_regs={'t': 883})
	Delay(t=0.5, auto=True, gens=True, ros=True, tag=None, t_params={'t': <qick.asm_v2.QickParam object at 0xffff7a50f400>}, t_regs={'t': 1536000192})
	IncReg(dst='s_core_w1', src=1)
	CloseLoo